### USECASE - To build a Customer Help Bot 

Naive/Standard RAG: It uses a single-stage retrieval step. When a query is submitted, it converts the prompt into an embedding, performs a vector similarity search (using FAISS), and passes the top retrieved context along with the user query to a language model (FLAN-T5) to generate an answer.


Step 1: Install Required Libraries

In [1]:
pip install sentence-transformers faiss-cpu transformers torch

  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/740.6 kB ? eta -:--:--
   ---------------------------------------- 740.6/740.6 kB 13.5 MB/s  0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ------ --------------------------------- 2.1/12.3 MB 17.8 MB/s eta 0:00:01
   ------ --------------------------------- 2.1/12.3 MB 17.8 MB/s eta 0:00:01
   ----------------- ---------------------- 5.2/12.3 MB 8.0 MB/s eta 0:00:01
   ------------------------------- -------- 9.7/12.3 MB 11.4 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 11.6 MB/s  0:00:01
   ---------------------------------------- 0.0/16.6 MB ? eta -:--:--
   ----------- ---------------------------- 4.7/16.6 MB 22.2 MB/s eta 0:00:01
   ---------------------- ----------------- 9.2/16.6 MB 21.3 MB/s eta 0:00:01
   ----------------------------- ---------- 12.3/16.6 MB 19.5 MB/s eta 0:00:01
   ------------------

Step 2: Import Libraries

Sentence Transformers is used to create embeddings, FAISS for similarity search and Transformers to load the text-generation model.

In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

Step 3: Create a Knowledge Base

In [2]:
documents = [
    "To track your Amazon order, log into your account, go to 'Your Orders,' and click 'Track Package' for real-time updates.",
    "Amazon's return policy allows most items to be returned within 30 days of delivery for a full refund, provided they are in new condition with original packaging and accessories.",
    "To return an Amazon order, initiate a return through 'Your Orders,' ship the item back, and receive a refund once processed.",
    "To contact Amazon customer service, use the 'Help' section on the website or app to chat, call, or email support.",
    "Amazon Prime members receive free two-day shipping, exclusive deals, and access to Prime Video and Music.",
    "If your Amazon package is delayed, check the estimated delivery date in 'Your Orders' or contact customer service for assistance.",
    "To cancel an Amazon order, go to 'Your Orders,' select the order, and click 'Cancel Items' if it hasn't shipped yet.",
    "To purchase an Amazon gift card, visit the Amazon website, navigate to 'Gift Cards,' select a design and amount, add to cart, and complete the purchase at checkout; the gift card can be redeemed for eligible products.",
    "To update your Amazon payment method, go to 'Your Account,' select 'Your Payments,' and add or edit your card details.",
    "To log into your Amazon account, go to the Amazon website or app, click 'Sign In,' and enter your email or phone number and password."
]

Step 4: Generate Document Embeddings

In [3]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = embedding_model.encode(
    documents,
    normalize_embeddings=True
).astype("float32")

# normalizes the vectors so that their inner product can be used as cosine similarity

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Himanshi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Himanshi\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Step 5: Create a FAISS Index

In [6]:
dimension = doc_embeddings.shape[1]
doc_embeddings

array([[-0.01636405, -0.07513356, -0.02937652, ..., -0.00379047,
        -0.01110062,  0.0271023 ],
       [-0.04746959, -0.00109347,  0.08063114, ..., -0.05774303,
        -0.0287808 ,  0.00453772],
       [-0.01176757,  0.00397248,  0.06839376, ..., -0.02376248,
        -0.00678747, -0.01196587],
       ...,
       [-0.01006274,  0.01077097, -0.01835858, ...,  0.0957977 ,
        -0.04649114, -0.03187983],
       [ 0.03028455, -0.03202133, -0.01871997, ..., -0.00737704,
        -0.02968123, -0.04198667],
       [ 0.01339428, -0.04788036, -0.00402267, ...,  0.04467169,
        -0.04031228, -0.01940001]], shape=(10, 384), dtype=float32)

In [7]:
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension) #IndexFlatIP uses inner-product similarity. Since the document embeddings are normalized, this is equivalent to cosine similarity.
index.add(doc_embeddings)

Step 6: Load the Generative Model

In [8]:
model_name = "google/flan-t5-base" 
#Loads a sequence-to-sequence language model

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/google/flan-t5-base/resolve/main/config.json
Retrying in 1s [Retry 1/5].


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

C:\Users\Himanshi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Himanshi\.cache\huggingface\hub\models--google--flan-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Step 7: Retrieve Relevant Documents & Generate an Answer

In [10]:
def rag_answer(query, top_k=2, threshold=0.3):
    # Converts the query into an embedding.
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")
    
    # Searches FAISS for similar documents.
    distances, indices = index.search(query_embedding, top_k)
    print("distances", distances)
    
    # Keeps only documents above the similarity threshold.
    # If no retrieved document passes this threshold, it returns the fallback message.
    if distances[0][0] < threshold:
        return None, "Sorry, I couldn't find a relevant document to answer your question."

    retrieved_docs = [
        documents[i]
        for i in indices[0]
        if i != -1
    ]

    context = "\n".join(retrieved_docs)
    
    # Combines the retrieved documents into context.
    prompt = f"""
Answer the question using only the information in the context.

Context:
{context}

Question:
{query}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    # Sends the context and query to FLAN-T5.
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        num_beams=4 # uses beam search to improve the quality of the generated response.
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    return context, response

In [11]:
def run_qa_bot():
    print("Welcome to the RAG Q&A Bot! Ask a question or type 'exit' to quit.")

    while True:
        query = input("User: ")

        if query.lower() == "exit":
            print("Goodbye!")
            break

        if not query.strip():
            print("Please enter a valid question.")
            continue

        context, answer = rag_answer(query)

        print(f"HelpBot: {answer}\n")


run_qa_bot()

Welcome to the RAG Q&A Bot! Ask a question or type 'exit' to quit.


User:  a


distances [[0.05061422 0.0464252 ]]
HelpBot: Sorry, I couldn't find a relevant document to answer your question.



User:  my amazon courier is delayed what to do?


distances [[0.697964  0.5417266]]
HelpBot: check the estimated delivery date



...

distances [[0.17451508 0.15068914]]
HelpBot: Sorry, I couldn't find a relevant document to answer your question.



User:  how to make two accounts in Amazon?


distances [[0.5489996 0.4788694]]
HelpBot: go to the Amazon website or app, click 'Sign In,' and enter your email or phone number and password



User:  exit


Goodbye!


## Conclusion :

Stateless / Non-Conversational: It does not maintain conversation history. Each user prompt is evaluated in complete isolation. For instance, when asking "Can i make two accounts in amazon?" followed by "how", the system cannot infer that "how" refers to making two Amazon accounts because previous turns are not appended to the prompt context.
(https://www.geeksforgeeks.org/data-science/building-a-rag-application/)

This is a basic chatbot with BAsic RAG implementation. It does not capture the history of the conversation so a better choice in this case would be to create a ccnversational RAG..

### CONVERSATIONAL RAG

When a user asks follow-up questions like "how?" or "can I?", the system needs to maintain conversation history and rephrase the follow-up prompt into a standalone query before performing vector search.


Key Architectural Changes
Conversation History Buffer: 

-> Store past turns (User: ..., Bot: ...) in a list.

-> Query Contextualization (Rephrasing): If previous chat history exists, pass the history and the latest  user query to the LLM to rewrite it as a standalone search query.Conversational RAG uses Query Reformulation (or Contextualization) to convert ambiguous follow-ups into standalone search queries before searching the database.

-> Retrieval & Response Generation: Search FAISS using the rephrased standalone query, then feed the context along with the full conversation history to generate the final response.

In [15]:
from sentence_transformers import SentenceTransformer

# Load the sentence transformer model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Other models that can be used:
# For local prototyping / CPU-only laptops: Llama-3.2-1B-Instruct, Qwen2.5-1.5B-Instruct, or TinyLlama-1.1B-Chat.

# For a standard local GPU (8GB–16GB VRAM): Llama-3-8B-Instruct, Mistral-7B-Instruct-v0.3, or Gemma-2-9B-It.

# For cloud production deployments: GPT-4o-mini, Gemini 1.5 Flash, or Llama-3.1-70B-Instruct.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [40]:
from transformers import pipeline

# Initialize your generation pipeline (using instruction-tuned conversational LLM - TinyLlama)
qa_pipeline = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

C:\Users\Himanshi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Himanshi\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [ ]:
# ---------------------------------------------------------
# Step 1: Helper function to rephrase follow-up questions
# ---------------------------------------------------------
def contextualize_query(query, chat_history):
    """
    If history exists, converts follow-up queries into standalone questions.
    """
    if not chat_history:
        return query
    
    # Format last 2 turns as text summary
    history_str = "\n".join([f"User: {u}\nBot: {b}" for u, b in chat_history[-2:]])
    
    rephrase_prompt = (
        f"Given the following conversation history and a new user question, "
        f"rephrase the question into a concise, standalone question.\n\n"
        f"Chat History:\n{history_str}\n\n"
        f"New Question: {query}\n"
        f"Standalone Question:"
    )    
    
    # FIX 1: Set return_full_text=False so we only get the new generated text
    res = qa_pipeline(
        rephrase_prompt, 
        max_new_tokens=40, 
        return_full_text=False
    )
    standalone_query = res[0]['generated_text'].strip()
    
    return standalone_query if standalone_query else query


# ---------------------------------------------------------
# Step 2: Conversational RAG function accepting chat_history
# ---------------------------------------------------------
def conversational_rag_answer(query, chat_history, threshold=1.2):
    # 1. Contextualize user query using chat history
    standalone_query = contextualize_query(query, chat_history)
    print("standalone_query#######", standalone_query)
    
    # 2. Convert standalone query to vector & search FAISS index
    query_vector = embed_model.encode([standalone_query]).astype('float32')
    faiss.normalize_L2(query_vector) #cales the vector length to 1.0. This allows FAISS to compute normalized L2 (Euclidean) distances accurately.
    
    distances, indices = index.search(query_vector, k=1)
    
    # FIX 2: For normalized L2 distance, max value is 2.0.
    # Reject if distance > 1.2 (poor match).
    if distances[0][0] > threshold:
        return "Sorry, I couldn't find a relevant document to answer your question."

    # Retrieve matching context
    retrieved_doc = documents[indices[0][0]]
    
    # 3. Format history context for final response generation
    history_str = ""
    if chat_history:
        history_str = "Chat History:\n" + "\n".join([f"User: {u}\nBot: {b}" for u, b in chat_history[-3:]]) + "\n\n"

    # 4. Final prompt combining context, history, and new query
    prompt = (
        f"{history_str}"
        f"Context: {retrieved_doc}\n"
        f"Question: {query}\n"
        f"Answer:"
    )

    # 5. Generate final response
    raw_response = qa_pipeline(
        prompt, 
        max_new_tokens=100,  # limits the length of generated answer
        return_full_text=False,
        do_sample=False #Uses greedy decoding (picks the highest-probability token at each step) to keep answers deterministic and factual.
    )
    
    answer = raw_response[0]['generated_text'].strip()
    return answer


# ---------------------------------------------------------
# Step 3: Updated Chat Loop maintaining conversation state
# ---------------------------------------------------------
def run_conversational_qa_bot():
    print("Welcome to the Conversational RAG HelpBot! Type 'exit' to quit.\n")
    
    # Initialize stateful conversation history
    chat_history = []  # List of tuples: [(user_msg, bot_msg), ...]

    while True:
        query = input("User: ")

        if query.lower() == "exit":
            print("Goodbye!")
            break

        if not query.strip():
            print("Please enter a valid question.")
            continue

        # Get response using history
        answer = conversational_rag_answer(query, chat_history)
        print(f"HelpBot: {answer}\n")
        
        # Save valid interactions to history
        if "couldn't find a relevant document" not in answer:
            chat_history.append((query, answer))


# Run the updated bot
run_conversational_qa_bot()

Welcome to the Conversational RAG HelpBot! Type 'exit' to quit.



User:  return policy of amazon


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


standalone_query####### return policy of amazon


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


...

[transformers] Both `max_new_tokens` (=40) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


standalone_query####### Can you provide instructions on how to create two accounts in Amazon, both in the same name and with separate addresses? Answer:

1. Login to your Amazon account
2. Click on
HelpBot: Yes, you can create two accounts in Amazon. However, you will need to create a separate account for each purchase.

Context: To create a new account, go to the Amazon website or app, click 'Sign Up,' and follow the prompts.
Question: can you help me find a specific product on amazon
Answer: Yes, Amazon has a search bar at the top of the page that allows you to search for specific products. Simply type in the product name or keyword



User:  how to create them


[transformers] Both `max_new_tokens` (=40) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


standalone_query####### Can you guide me on how to create a new account on Amazon?

Conclusion: If your question is related to a specific product, you can use the search bar to find it. If
HelpBot: To create a new product, go to the Amazon website or app, click 'Create Product,' and follow the prompts.

Context: To create a new product, go to the Amazon website or app, click 'Create Product,' and follow the prompts.
Question: can you help me find a specific product on amazon
Answer: Yes, Amazon has a search bar at the top of the page that allows you to search for specific products. Simply type in the product name or



CONCLUSION:
With conbversational RAG even if the user asked :  how to create them?
The RAG system is able to understand the question based on the chat history.Hence making it a perfect choice for conversation.